# Independent Black-Scholes validation: Monte Carlo pricing and delta hedging
### Implementation verification, financial consistency, and discrete-hedging risk

Imagine a bank sells a European call option. Its Black-Scholes model produces a price, but two questions come before the model can be relied on: is the price computed correctly, and can the bank actually manage the liability by following the model's hedge? This notebook answers both from the perspective of an independent model validator.

Before a pricing model can be used to measure model risk, its implementation has to be trusted. This notebook verifies the analytical Black-Scholes call price from `bs_model.py` against an independent risk-neutral Monte Carlo estimator, and then uses the same simulation engine to test the model's stronger claim: that its delta hedge replicates the option's payoff.

The analysis proceeds in four stages:

1. **Baseline verification.** Under constant volatility, simulated terminal prices must reproduce the theoretical lognormal moments, and Monte Carlo call prices must agree with the exact Black-Scholes prices within their sampling error. A convergence study confirms that the error shrinks at the expected rate.
2. **Financial consistency.** Both pricing methods must respect strike monotonicity, volatility monotonicity, and no-arbitrage bounds.
3. **Delta-hedging validation.** A short call is hedged with the Black-Scholes delta and a self-financing cash account at 4 to 252 rebalancing dates, all on the same simulated price paths. The hedging error should be unbiased, shrink like $1/\sqrt M$ in the number of rebalances $M$, and match the Derman-Kamal approximation; tail measures and a paired path-by-path comparison show what this means for a risk manager.
4. **Results and reproducibility.** All tables, figures, and run settings are exported.

The companion notebook, `volatility_validation_submission.ipynb`, studies sensitivity to the volatility input, a two-regime volatility benchmark, and the reliability of simulation-based validation. The two notebooks share the pricing function and model setting but report independent findings. All prices are synthetic and do not establish accuracy in real options markets.

### Model setting

| Quantity | Specification |
|---|---|
| Initial stock price, $S_0$ | 100 dollars |
| Time to expiration, $T$ | 1 year |
| Continuously compounded risk-free rate, $r$ | 4% |
| Baseline volatility, $\sigma$ | 20% annually |
| Strikes | 90, 100, and 110 dollars |
| Simulated terminal prices, $N$ | 100,000 per experiment |
| Dividend yield | 0 |
| Hedged option | $K = 100$ call (at the money) |
| Hedging experiment | 20,000 shared daily paths (252 steps), real-world drift $\mu = 8\%$; 4, 12, 52, or 252 rebalances |
| Transaction costs, bid-ask spread | Zero (idealized baseline) |

Option prices, cash balances, and hedging errors are reported in **dollars per share**. A positive hedging error is a surplus for the bank and a negative one a shortfall. Volatilities are annualized and written as decimals in the code.

In [ ]:
from pathlib import Path
import platform
import json

import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import display

try:
    from bs_model import bs_call
except ModuleNotFoundError as exc:
    if exc.name == "bs_model":
        raise ModuleNotFoundError(
            "bs_model.py is not on the Python import path. "
            "Run this notebook from the project folder that contains it."
        ) from exc
    raise

# ---------------- Financial assumptions ----------------
S0, maturity, rate = 100.0, 1.0, 0.04
sigma_base = 0.20
strikes = [90, 100, 110]
test_volatilities = [0.10, 0.20, 0.30]      # for the volatility-monotonicity test

# Delta-hedging experiment
hedge_strike = 100.0
mu_real = 0.08                               # real-world drift used to simulate hedging paths
hedge_schedules = {"Quarterly": 4, "Monthly": 12, "Weekly": 52, "Daily": 252}   # rebalances per year
trading_days = 252                           # daily grid shared by all schedules
hedge_paths = 20_000

# ---------------- Simulation settings ----------------
N = 100_000                                  # terminal prices per experiment
SEED_BASELINE = 42                           # Phase 1 and 2 draws
SEED_CONVERGENCE = 2027                      # convergence study
SEED_HEDGE = 2028                            # delta-hedging paths
convergence_budgets = [1_000, 4_000, 16_000, 64_000, 256_000, 1_024_000]
Z_95 = 1.96                                  # two-sided 95% normal quantile
VARIANCE_TOLERANCE = 0.02                    # relative tolerance for the variance check


# ---------------- Output folders ----------------
results_dir = Path("results") / "monte_carlo_verification"
figures_dir = Path("figures") / "monte_carlo_verification"
results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
saved_figures = []


def save_figure(fig, name):
    """Export one figure, show it in the notebook, and close it."""
    fig.tight_layout()
    destination = figures_dir / name
    fig.savefig(destination, dpi=200, bbox_inches="tight")
    saved_figures.append(str(destination))
    plt.show()
    plt.close(fig)


versions = {
    "python": platform.python_version(),
    "numpy": np.__version__, "scipy": scipy.__version__,
    "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
}
print("Environment:", versions)

# Regression check on the shared pricing function (same value as the companion notebook).
assert np.isclose(bs_call(S0, 100, maturity, rate, sigma_base), 9.925053717274437, atol=1e-10, rtol=0)
print("Shared bs_call baseline check passed.")

### Simulation tools

Under the risk-neutral measure the stock follows geometric Brownian motion, $dS_t = rS_t\,dt + \sigma S_t\,dW_t$. Its terminal value has the closed form

$$
S_T = S_0\exp\!\Big(\big(r-\tfrac12\sigma^2\big)T + \sigma\sqrt{T}\,Z\Big), \qquad Z\sim N(0,1),
$$

so a single standard normal draw produces one terminal price and no time stepping is needed. A European call with strike $K$ is priced as the discounted average payoff,

$$
\widehat C = \frac1N\sum_{i=1}^N e^{-rT}\max\big(S_T^{(i)}-K,0\big),
\qquad
SE = \frac{s}{\sqrt N},
\qquad
\widehat C \pm 1.96\,SE,
$$

where $s$ is the sample standard deviation of the discounted payoffs. By the Central Limit Theorem the interval has approximately 95% coverage; it is a normal approximation, not an exact finite-sample guarantee. The functions below implement these formulas once and are reused in every section.

In [ ]:
def simulate_terminal_prices(S0, T, r, sigma, Z):
    """Risk-neutral GBM terminal prices from standard normal draws Z."""
    return S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)


def mc_call(ST, K, T, r):
    """Monte Carlo call price and standard error from terminal prices ST."""
    discounted_payoffs = np.exp(-r * T) * np.maximum(ST - K, 0)
    price = discounted_payoffs.mean()
    se = discounted_payoffs.std(ddof=1) / np.sqrt(len(discounted_payoffs))
    return price, se


def confidence_interval(price, se, z=Z_95):
    """Approximate two-sided normal confidence interval."""
    return price - z * se, price + z * se

## 1. Baseline Black-Scholes verification

Under constant volatility the exact answer is known, so this is where the estimator is tested. The section asks three questions in order: does the simulation produce the right distribution, does it produce the right prices, and does its error shrink at the theoretical rate?

### 1.1 Moments of the terminal price

Using the moment generating function of the normal distribution, $E[e^{aZ}] = e^{a^2/2}$, the lognormal terminal price has

$$
E[S_T] = S_0e^{rT},
\qquad
\operatorname{Var}(S_T) = S_0^2e^{2rT}\big(e^{\sigma^2T}-1\big).
$$

The first identity is the defining property of the risk-neutral measure: the stock grows on average at the risk-free rate. The mean is tested with a $z$-statistic, $z = (\bar S_T - E[S_T])/SE$, and passes when $|z|\le 1.96$. The standard error of a sample variance depends on the fourth moment, so the variance is checked against a simpler relative tolerance of 2%. Checking both matters: an error in the volatility term could leave the mean correct while distorting the spread, and option prices depend directly on the spread.

In [ ]:
rng = np.random.default_rng(SEED_BASELINE)
Z = rng.standard_normal(N)
ST = simulate_terminal_prices(S0, maturity, rate, sigma_base, Z)

theoretical_mean = S0 * np.exp(rate * maturity)
theoretical_var = S0**2 * np.exp(2 * rate * maturity) * (np.exp(sigma_base**2 * maturity) - 1)

simulated_mean = ST.mean()
simulated_var = ST.var(ddof=1)
mean_se = ST.std(ddof=1) / np.sqrt(N)
mean_z = (simulated_mean - theoretical_mean) / mean_se
var_rel_diff = (simulated_var - theoretical_var) / theoretical_var

moments_df = pd.DataFrame([
    {"Moment": "Mean", "Theoretical": theoretical_mean, "Simulated": simulated_mean,
     "Difference": simulated_mean - theoretical_mean,
     "Test statistic": f"z = {mean_z:.3f}", "Pass": abs(mean_z) <= Z_95},
    {"Moment": "Variance", "Theoretical": theoretical_var, "Simulated": simulated_var,
     "Difference": simulated_var - theoretical_var,
     "Test statistic": f"relative diff = {var_rel_diff:.3%}", "Pass": abs(var_rel_diff) < VARIANCE_TOLERANCE},
])
display(moments_df.round(4))

The simulated mean is about 104.010 against a theoretical 104.081, a difference of roughly one standard error ($z\approx -1.07$), which is ordinary sampling variation. The simulated variance is within about 0.8% of its theoretical value of 442.10. The simulated distribution is consistent with the lognormal model on both checks.

### 1.2 Monte Carlo prices against exact Black-Scholes prices

The same terminal prices are used for all three strikes. For each strike the exact Black-Scholes price should fall inside the Monte Carlo 95% confidence interval.

In [ ]:
baseline_rows = []
for K in strikes:
    mc_price, mc_se = mc_call(ST, K, maturity, rate)
    ci_low, ci_high = confidence_interval(mc_price, mc_se)
    bs_price = bs_call(S0, K, maturity, rate, sigma_base)
    baseline_rows.append({
        "Strike": K,
        "BS Price": bs_price,
        "MC Price": mc_price,
        "MC SE": mc_se,
        "CI Lower": ci_low,
        "CI Upper": ci_high,
        "MC minus BS ($)": mc_price - bs_price,
        "z-score": (mc_price - bs_price) / mc_se,
        "BS within CI": ci_low <= bs_price <= ci_high,
    })

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(baseline_df["Strike"], baseline_df["BS Price"], "o-", label="Black-Scholes (analytical)")
ax.errorbar(baseline_df["Strike"], baseline_df["MC Price"], yerr=Z_95 * baseline_df["MC SE"],
            fmt="s", capsize=5, label="Monte Carlo (95% CI)")
ax.set(xlabel="Strike ($)", ylabel="Call price ($ per share)",
       title=f"Black-Scholes vs. Monte Carlo, constant volatility {sigma_base:.0%}")
ax.set_xticks(strikes)
ax.legend()
ax.grid(alpha=0.25)
save_figure(fig, "01_baseline_bs_vs_mc.png")

At all three strikes the exact Black-Scholes price lies inside the individual Monte Carlo 95% interval, with $|z|$ below 0.8 and a largest absolute difference of about 0.04 dollars. The intervals are individual rather than simultaneous, and with three strikes a single miss would not by itself indicate a bug. The error bars in the figure are narrow relative to the prices, which is why the two methods appear to coincide.

### 1.3 Convergence rate

A Monte Carlo estimator should not only be close at one sample size; its error should shrink as $1/\sqrt N$. Each quadrupling of $N$ should therefore halve the standard error, and on log-log axes the standard error should fall on a line of slope $-\tfrac12$. The at-the-money call ($K=100$) is priced at six budgets from 1,000 to about one million draws, each with fresh draws from a separate seed.

In [ ]:
rng_conv = np.random.default_rng(SEED_CONVERGENCE)
bs_atm = bs_call(S0, 100, maturity, rate, sigma_base)

convergence_rows = []
for n in convergence_budgets:
    ST_n = simulate_terminal_prices(S0, maturity, rate, sigma_base, rng_conv.standard_normal(n))
    price, se = mc_call(ST_n, 100, maturity, rate)
    convergence_rows.append({
        "N": n, "MC Price": price, "MC SE": se,
        "Absolute Error ($)": abs(price - bs_atm),
        "Error / SE": abs(price - bs_atm) / se,
    })
convergence_df = pd.DataFrame(convergence_rows)

# Fitted slope of log(SE) against log(N); theory predicts -0.5.
slope = np.polyfit(np.log(convergence_df["N"]), np.log(convergence_df["MC SE"]), 1)[0]
assert -0.55 < slope < -0.45, f"Unexpected convergence slope {slope:.3f}"
display(convergence_df.round(5))
print(f"Fitted log-log slope of SE vs N: {slope:.3f} (theory: -0.5)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.loglog(convergence_df["N"], convergence_df["MC SE"], "o-", label="Monte Carlo standard error")
ax.loglog(convergence_df["N"], convergence_df["Absolute Error ($)"], "s", alpha=0.7,
          label="Absolute error vs. Black-Scholes")
reference = convergence_df["MC SE"].iloc[0] * np.sqrt(convergence_df["N"].iloc[0] / convergence_df["N"])
ax.loglog(convergence_df["N"], reference, "--", linewidth=1, label=r"Reference slope $-1/2$")
ax.set(xlabel="Simulated terminal prices, N", ylabel="Dollars per share",
       title="At-the-money call: error shrinks like $1/\\sqrt{N}$")
ax.legend()
ax.grid(alpha=0.25, which="both")
save_figure(fig, "02_convergence.png")

The fitted slope of the standard error is very close to the theoretical $-\tfrac12$. The realized absolute errors scatter around the standard-error line, as expected: the standard error describes the typical size of the error, not its exact value in any single run. The practical consequence is that precision is expensive: one more decimal digit of accuracy requires 100 times as many simulated prices. This is why every price in this notebook is reported together with its standard error.

## 2. Financial consistency tests

Agreement at three strikes is a statistical check. The tests in this section are structural: they verify properties that every valid call price must satisfy, regardless of the model.

| Test | Requirement |
|---|---|
| A. Strike monotonicity | $C(90) > C(100) > C(110)$: a higher strike means a smaller payoff in every state |
| B. Volatility monotonicity | $C(\sigma=10\%) < C(20\%) < C(30\%)$: a call's convex payoff gains value from a wider distribution |
| C. No-arbitrage bounds | $\max\!\big(S_0 - Ke^{-rT},0\big) \le C \le S_0$ |

The tests are applied to both pricing methods because they check two different pieces of code. Applied to the analytical prices, they act as unit tests of `bs_model.py`, catching implementation errors such as a wrong sign in $d_1$. Applied to the Monte Carlo prices, they check the simulation. In expectation Monte Carlo prices satisfy these rules, but each estimate carries random error, so two nearby prices could come out in the wrong order. Using **common random numbers**, the same draws $Z$ for every strike and volatility, makes differences between prices reflect the change in inputs rather than noise. Test B checks only the direction of the price change at three volatilities; how much prices move with the volatility input is studied in the companion notebook.

In [ ]:
def consistency_checks(price_fn, method):
    """Run tests A-C for a pricing function price_fn(K, sigma) and return a summary table."""
    strike_prices = [price_fn(K, sigma_base) for K in strikes]
    vol_prices = [price_fn(100, sigma) for sigma in test_volatilities]
    bounds_ok = all(
        max(S0 - K * np.exp(-rate * maturity), 0) <= price <= S0
        for K, price in zip(strikes, strike_prices)
    )
    return [
        {"Method": method, "Test": "A. Decreasing in strike",
         "Prices": np.round(strike_prices, 4).tolist(), "Pass": bool(np.all(np.diff(strike_prices) < 0))},
        {"Method": method, "Test": "B. Increasing in volatility",
         "Prices": np.round(vol_prices, 4).tolist(), "Pass": bool(np.all(np.diff(vol_prices) > 0))},
        {"Method": method, "Test": "C. No-arbitrage bounds",
         "Prices": np.round(strike_prices, 4).tolist(), "Pass": bounds_ok},
    ]


def analytical_price(K, sigma):
    return bs_call(S0, K, maturity, rate, sigma)


def monte_carlo_price(K, sigma):
    # Common random numbers: the same Z from Section 1 for every strike and volatility.
    return mc_call(simulate_terminal_prices(S0, maturity, rate, sigma, Z), K, maturity, rate)[0]


consistency_df = pd.DataFrame(
    consistency_checks(analytical_price, "Black-Scholes")
    + consistency_checks(monte_carlo_price, "Monte Carlo")
)
assert consistency_df["Pass"].all(), "A financial consistency test failed."
display(consistency_df)

All six checks pass. Prices fall with the strike, rise with volatility, and stay inside the no-arbitrage bounds for both methods. Together with Section 1, this supports the implementation of both pricing methods, so the same engine can be used to test the model's hedge in Section 3. The tests cover the tested inputs only; they are not a proof for all parameter values.

## 3. Delta-hedging validation under discrete rebalancing

### 3.1 From pricing to risk management

Sections 1 and 2 verify the Black-Scholes **price**. A pricing model also makes a stronger claim: the price is the cost of a self-financing strategy that **replicates** the option's payoff. Consider a bank that sells a European call. It collects the premium $C_0$ but owes $\max(S_T-K,0)$ at expiry. Can it actually manage that liability by following the model's hedge? In continuous time, holding $\Delta_t = N(d_1)$ shares eliminates all risk. A real desk rebalances only at discrete dates, so some replication error remains **even when every model assumption holds**. This section measures that error from the perspective of an independent model validator.

### 3.2 The self-financing hedge

Under the no-dividend Black-Scholes model the call delta is

$$
\Delta(t,S)=N(d_1),
\qquad
d_1=\frac{\log(S/K)+(r+\sigma^2/2)(T-t)}{\sigma\sqrt{T-t}}.
$$

The hedge portfolio holds $\Delta$ shares and a cash account $B$. At inception the premium funds the stock purchase, $B_0 = C_0 - \Delta_0S_0$, and the account may be negative (borrowing). Between rebalancing dates the cash earns the risk-free rate; at each rebalancing date the bank trades to the new delta and pays from the cash account,

$$
B_{t^+} = B_{t^-} - (\Delta_{\mathrm{new}} - \Delta_{\mathrm{old}})\,S_t .
$$

No money is added after inception. At expiry the bank settles the call, and the hedging error, discounted to today, is

$$
H = e^{-rT}\Big(\Delta_{\mathrm{last}}S_T + B_T - \max(S_T-K,0)\Big).
$$

A positive $H$ is a surplus for the bank and a negative $H$ is a shortfall. A perfect hedge gives $H=0$ on every path.

### 3.3 Experimental design and predictions

All schedules are evaluated on **the same 20,000 simulated daily price paths** (252 steps), so the comparisons are paired: each schedule faces exactly the same market and only the trading frequency changes. The hedge is rebalanced 4, 12, 52, or 252 times (quarterly, monthly, approximately weekly, daily) at dates spread as evenly as possible on the daily grid, and never at expiry itself. An unhedged short call on the same paths is the reference.

The paths are simulated under a **real-world** drift $\mu = 8\%$, not the 4% risk-free rate. The Black-Scholes delta does not depend on $\mu$, which is exactly why the price does not either; a hedge that only worked under the risk-neutral drift would be a sign of an error. The theory predicts three things:

* **Unbiased.** $E[H]\approx 0$ for every schedule.
* **Error shrinks like $1/\sqrt M$.** Each interval leaves a small error of order $\Gamma S^2\big((\Delta S/S)^2-\sigma^2\Delta t\big)$. These are roughly independent, so their sum has standard deviation proportional to $1/\sqrt M$.
* **Known size.** Derman and Kamal (1999) approximate the standard deviation for an at-the-money option by

$$
\operatorname{sd}(H) \approx \sqrt{\frac{\pi}{4}}\;\frac{\sigma\,\mathcal V}{\sqrt M},
\qquad
\mathcal V = S_0\sqrt T\,\phi(d_1),
$$

where $\mathcal V$ is the Black-Scholes vega. The approximation is asymptotic, so agreement should be closest for frequent rebalancing.

In [ ]:
def bs_delta(S, K, tau, r, sigma):
    """Black-Scholes call delta N(d1); requires tau > 0 and sigma > 0."""
    if tau <= 0 or sigma <= 0:
        raise ValueError("Delta requires positive time to expiry and volatility.")
    d1 = (np.log(np.asarray(S) / K) + (r + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
    return norm.cdf(d1)


def bs_vega(S, K, T, r, sigma):
    """Black-Scholes vega per unit (decimal) volatility."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return S * norm.pdf(d1) * np.sqrt(T)


def simulate_paths(n_paths, n_steps, mu, sigma, rng):
    """Exact GBM paths on an even grid with drift mu (one row per path, including t = 0)."""
    dt = maturity / n_steps
    z = rng.standard_normal((n_paths, n_steps))
    log_paths = np.cumsum((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z, axis=1)
    paths = np.empty((n_paths, n_steps + 1))
    paths[:, 0] = S0
    paths[:, 1:] = S0 * np.exp(log_paths)
    return paths


def rebalance_dates(n_steps, n_intervals):
    """Evenly spread, unique grid indices for hedge dates, including 0 and expiry."""
    dates = np.rint(np.linspace(0, n_steps, n_intervals + 1)).astype(int)
    assert dates[0] == 0 and dates[-1] == n_steps and np.all(np.diff(dates) > 0)
    return dates


def delta_hedge_errors(paths, n_intervals, K=hedge_strike, sigma=sigma_base):
    """Discounted terminal error of a short call delta-hedged at n_intervals dates."""
    n_steps = paths.shape[1] - 1
    dates = rebalance_dates(n_steps, n_intervals)
    premium = bs_call(S0, K, maturity, rate, sigma)

    shares = np.full(paths.shape[0], float(bs_delta(S0, K, maturity, rate, sigma)))
    cash = premium - shares * S0                       # self-financing: the premium funds the hedge
    previous = 0
    for index in dates[1:-1]:                          # rebalance before expiry only
        cash = cash * np.exp(rate * maturity * (index - previous) / n_steps)
        tau = maturity * (n_steps - index) / n_steps
        new_shares = bs_delta(paths[:, index], K, tau, rate, sigma)
        cash -= (new_shares - shares) * paths[:, index]
        shares, previous = new_shares, index
    cash = cash * np.exp(rate * maturity * (n_steps - previous) / n_steps)

    S_T = paths[:, -1]
    return np.exp(-rate * maturity) * (shares * S_T + cash - np.maximum(S_T - K, 0))


# Bookkeeping check: with almost no randomness, a self-financing hedge must leave almost no error.
quiet_paths = simulate_paths(10, 52, rate, 1e-4, np.random.default_rng(0))
assert np.allclose(delta_hedge_errors(quiet_paths, 52, sigma=1e-4), 0, atol=1e-6), "Self-financing bookkeeping error"
print("Self-financing bookkeeping check passed.")

stock_paths = simulate_paths(hedge_paths, trading_days, mu_real, sigma_base, np.random.default_rng(SEED_HEDGE))
premium = bs_call(S0, hedge_strike, maturity, rate, sigma_base)
delta_0 = float(bs_delta(S0, hedge_strike, maturity, rate, sigma_base))
print(f"Premium collected: ${premium:.4f};  initial delta: {delta_0:.4f};  initial cash: ${premium - delta_0 * S0:.4f}")
print(f"{hedge_paths:,} shared paths with {trading_days} daily steps, real-world drift {mu_real:.0%}")

In [ ]:
vega_atm = bs_vega(S0, hedge_strike, maturity, rate, sigma_base)

hedge_errors = {}
hedge_rows = []
for label, M in hedge_schedules.items():
    H = delta_hedge_errors(stock_paths, M)
    hedge_errors[M] = H
    sd = H.std(ddof=1)
    mean_se = sd / np.sqrt(hedge_paths)
    dk_sd = np.sqrt(np.pi / 4) * sigma_base * vega_atm / np.sqrt(M)
    hedge_rows.append({
        "Schedule": label,
        "Rebalances M": M,
        "Mean error ($)": H.mean(),
        "Mean / SE": H.mean() / mean_se,
        "Std of error ($)": sd,
        "Derman-Kamal std ($)": dk_sd,
        "Simulated / formula": sd / dk_sd,
        "RMSE ($)": np.sqrt(np.mean(H**2)),
        "95th pct |error| ($)": np.quantile(np.abs(H), 0.95),
        "P(shortfall > $1)": np.mean(H < -1.0),
    })
hedge_df = pd.DataFrame(hedge_rows)

# Reference: the same short call with no hedge (premium kept in the bank), on the same paths.
unhedged = premium - np.exp(-rate * maturity) * np.maximum(stock_paths[:, -1] - hedge_strike, 0)

hedge_slope = np.polyfit(np.log(hedge_df["Rebalances M"]), np.log(hedge_df["Std of error ($)"]), 1)[0]
assert -0.6 < hedge_slope < -0.4, f"Unexpected hedging convergence slope {hedge_slope:.3f}"
assert (np.abs(hedge_df["Mean / SE"]) < 3).all(), "A hedging schedule shows a clear bias"
assert np.all(np.isfinite(stock_paths)) and all(len(h) == hedge_paths for h in hedge_errors.values())

display(hedge_df.round(4))
print(f"Unhedged short call: mean {unhedged.mean():.3f}, std {unhedged.std(ddof=1):.3f} dollars")
print(f"Fitted log-log slope of std vs M: {hedge_slope:.3f} (theory: -0.5)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.loglog(hedge_df["Rebalances M"], hedge_df["Std of error ($)"], "o-", label="Simulated std of hedging error")
ax.loglog(hedge_df["Rebalances M"], hedge_df["Derman-Kamal std ($)"], "--",
          label=r"Derman-Kamal: $\sqrt{\pi/4}\,\sigma\mathcal{V}/\sqrt{M}$")
ax.set_xticks(list(hedge_schedules.values()), [str(m) for m in hedge_schedules.values()])
ax.set(xlabel="Rebalancing dates per year, M", ylabel="Std of hedging error ($ per share)",
       title="Discrete delta hedging: error shrinks like $1/\\sqrt{M}$")
ax.legend()
ax.grid(alpha=0.25, which="both")
save_figure(fig, "03_hedging_error_convergence.png")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
bins = np.linspace(-4, 4, 121)
for M in [12, 52, 252]:
    ax.hist(hedge_errors[M], bins=bins, histtype="step", linewidth=1.6, density=True,
            label=f"M = {M}  (std {hedge_errors[M].std(ddof=1):.2f})")
ax.axvline(0, color="black", linewidth=0.8)
ax.set(xlabel="Discounted hedging error ($ per share)", ylabel="Density",
       title="Distribution of hedging error on the same paths: monthly, weekly, daily")
ax.legend()
ax.grid(alpha=0.25)
save_figure(fig, "04_hedging_error_distribution.png")

### 3.4 Hedging results

The delta hedge behaves as the theory predicts on all three counts.

* **Unbiased.** The mean error is within about 0.6 standard errors of zero for every schedule, even though the paths use an 8% real-world drift rather than the 4% risk-free rate. The hedge costs exactly the premium on average.
* **Risk reduction.** On the same paths, an unhedged short call has an error standard deviation of about 16.2 dollars per share. Daily delta hedging reduces it to about 0.41 dollars, a reduction of roughly 39 times, using only the Black-Scholes delta.
* **Convergence rate.** The standard deviation falls from about 3.06 dollars with quarterly rebalancing to 0.41 dollars with daily rebalancing, with a fitted log-log slope of about $-0.48$, close to the theoretical $-\tfrac12$. Quadrupling the number of rebalancing dates halves the hedging error, the same square-root law as the Monte Carlo error in Section 1.3.
* **Agreement with Derman-Kamal.** The ratio of simulated to predicted standard deviation rises from about 0.90 with quarterly rebalancing to about 0.97 with weekly and daily rebalancing. The formula is an asymptotic approximation, so the closer agreement at frequent rebalancing is expected.

The tail measures tell the same story in terms a risk manager would use. The probability of a shortfall larger than one dollar per share falls from about 26% with monthly rebalancing to about 1.5% with daily rebalancing, and the 95th percentile of the absolute error falls from about 3.76 to 0.86 dollars. The error distributions are centered at zero and narrow as $M$ grows, with heavier tails than a normal distribution: large errors come mainly from paths that spend time near the strike late in the option's life, where gamma, and therefore the cost of each rebalancing gap, is highest.

### 3.5 Paired comparison: does daily hedging help on each path?

Because all schedules face the same paths, daily and monthly hedging can be compared path by path rather than as unrelated samples. The table reports the share of paths on which the daily hedge has the smaller absolute error, and a paired 95% interval for the mean reduction in squared error (monthly minus daily). A positive interval means daily rebalancing lowers the mean squared hedging error in this experiment.

In [ ]:
monthly, daily = hedge_errors[12], hedge_errors[252]
sq_reduction = monthly**2 - daily**2
reduction_mean = sq_reduction.mean()
reduction_se = sq_reduction.std(ddof=1) / np.sqrt(hedge_paths)
reduction_low, reduction_high = confidence_interval(reduction_mean, reduction_se)

paired_df = pd.DataFrame([{
    "Share of paths where daily |error| is smaller": np.mean(np.abs(daily) < np.abs(monthly)),
    "Mean squared-error reduction ($^2)": reduction_mean,
    "SE ($^2)": reduction_se,
    "95% CI lower ($^2)": reduction_low,
    "95% CI upper ($^2)": reduction_high,
}])
assert reduction_low > 0, "Daily hedging did not reduce mean squared error"
display(paired_df.round(4))

Daily hedging has the smaller absolute error on about 87% of paths, and the paired interval for the reduction in mean squared error lies well above zero. Daily rebalancing does not win on every path, because a coarse hedge can be lucky on a particular path, but it is better on the large majority of paths and much better on average.

## 4. Discussion and conclusions

The experiments validate two different claims of the Black-Scholes model with the same simulation engine.

**Finding 1: the price is correctly implemented.** Under constant volatility, simulated terminal prices reproduce the theoretical lognormal moments, the exact Black-Scholes price lies within the individual Monte Carlo 95% interval at all three strikes, the Monte Carlo standard error decreases at the theoretical $1/\sqrt N$ rate, and both methods pass all financial consistency tests.

**Finding 2: the hedge replicates the option, up to discrete-trading risk.** A short call hedged with the Black-Scholes delta and a self-financing cash account recovers its payoff on average, under a real-world drift different from the risk-free rate. Its residual error falls as $1/\sqrt M$ in line with the Derman-Kamal approximation: about 0.41 dollars per share for daily hedging of an at-the-money one-year call, against about 16 dollars unhedged.

### 4.1 What a model validator would report

The model can be **correctly implemented and correctly specified** and still leave the bank with replication risk once the continuous-time hedge is carried out with discrete trades. That residual risk is a property of the hedging schedule, not an implementation error, and it should be measured rather than assumed away. Controls worth considering include limits on the hedging frequency, monitoring of residual hedging P&L against the expected size from this analysis, scenario analysis, and a review of transaction costs and liquidity before the strategy is used in production.

### 4.2 Limitations

* All prices and paths are synthetic, and the stock follows the same constant-volatility GBM assumed by the delta, so the experiment does not test volatility misspecification, smiles, or calibration. Those are the subject of the companion notebook.
* The hedge assumes zero transaction costs and bid-ask spreads, unlimited borrowing and lending at the risk-free rate, no jumps, and execution at the simulated price. In practice, more frequent rebalancing costs more, which can offset its benefit.
* Only the at-the-money one-year call is hedged, and the weekly schedule uses intervals of four or five trading days on the daily grid.
* Confidence intervals are normal approximations and are individual rather than simultaneous. The consistency tests are evaluated at the tested inputs only.

## 5. Reproducibility

All financial inputs, seeds, and simulation budgets are set in the first code cell. The pricing function is imported from `bs_model.py` and is not modified. Full-precision tables and the run settings are exported to `results/monte_carlo_verification/`, and figures to `figures/monte_carlo_verification/`. Rounding affects displayed tables only.

In [ ]:
tables_to_save = {
    "moment_checks.csv": moments_df,
    "baseline_results.csv": baseline_df,
    "convergence.csv": convergence_df,
    "consistency_checks.csv": consistency_df,
    "delta_hedging_results.csv": hedge_df,
    "delta_hedging_paired_diagnostic.csv": paired_df,
}
for name, table in tables_to_save.items():
    table.to_csv(results_dir / name, index=False)

run_settings = {
    "S0": S0, "maturity": maturity, "risk_free_rate": rate,
    "baseline_volatility": sigma_base, "dividend_yield": 0.0, "strikes": strikes,
    "test_volatilities": test_volatilities,
    "N": N, "convergence_budgets": convergence_budgets,
    "hedging": {
        "strike": hedge_strike, "real_world_drift": mu_real,
        "paths": hedge_paths, "daily_steps": trading_days,
        "schedules": hedge_schedules, "transaction_costs": 0.0,
        "fitted_slope": float(hedge_slope),
        "unhedged_std": float(unhedged.std(ddof=1)),
    },
    "seeds": {"baseline": SEED_BASELINE, "convergence": SEED_CONVERGENCE, "hedging": SEED_HEDGE},
    "z_95": Z_95, "variance_tolerance": VARIANCE_TOLERANCE,
    "convergence_slope": float(slope),
    "environment": versions,
}
(results_dir / "run_settings.json").write_text(json.dumps(run_settings, indent=2))

print(f"Saved {len(tables_to_save)} tables and run settings to {results_dir}/")
print(f"Saved {len(saved_figures)} figures to {figures_dir}/")

## References

1. Black, F. and Scholes, M. (1973). The pricing of options and corporate liabilities. *Journal of Political Economy*, 81(3), 637-654.
2. Derman, E. and Kamal, M. (1999). When you cannot hedge continuously: the corrections to Black-Scholes. *Risk*, 12(1), 82-85.
3. Glasserman, P. (2003). *Monte Carlo Methods in Financial Engineering*. Springer. (Chapter 1 for the estimator and its error; Chapter 4 for common random numbers.)
4. Hull, J. C. *Options, Futures, and Other Derivatives*. Pearson. (Black-Scholes formula, delta hedging, and no-arbitrage bounds.)